# Imports & Setup

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, balanced_accuracy_score, confusion_matrix
from joblib import dump

# 1. Load Raw Data

In [2]:
df = pd.read_csv("Data.csv")

print("Raw data shape:", df.shape)
df.head(3)

Raw data shape: (1742, 84)


,Flow ID,Src IP,Src Port,Dst IP,Dst Port,Protocol,Timestamp,Flow Duration,Total Fwd Packet,Total Bwd packets,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,10.0.2.15-10.0.2.5-39023-5432-6,10.0.2.15,39023,10.0.2.5,5432,6,15/12/2025 11:05:30 PM,19434,6,5,...,32,0,0,0,0,0.0,0.0,0.0,0.0,1
1,10.0.2.15-10.0.2.5-40639-5432-6,10.0.2.15,40639,10.0.2.5,5432,6,15/12/2025 11:05:31 PM,22348,5,5,...,32,0,0,0,0,0.0,0.0,0.0,0.0,0
2,10.0.2.15-10.0.2.5-44221-5432-6,10.0.2.15,44221,10.0.2.5,5432,6,15/12/2025 11:05:31 PM,22230,6,5,...,32,0,0,0,0,0.0,0.0,0.0,0.0,1


# 2. Data Cleaning & Preprocessing

In [3]:
# Drop identifiers (Flow ID, IPs, Timestamp)
drop_cols = ["Flow ID", "Src IP", "Dst IP", "Timestamp"]
df.drop(columns=drop_cols, inplace=True, errors="ignore")

# Replace Inf/-Inf with NaN
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Drop rows with too many NaNs
df.dropna(thresh=int(df.shape[1] * 0.8), inplace=True)

# Fill remaining NaNs with median
df.fillna(df.median(numeric_only=True), inplace=True)

# Separate features and labels
X = df.drop("Label", axis=1)
y = df["Label"]

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Save scaler for later use
dump(scaler, "results/scaler.joblib")

# Save processed DataFrame (optional)
processed = pd.DataFrame(X_scaled, columns=X.columns)
processed["Label"] = y.values
processed.to_csv("Data_clean.csv", index=False)

print("Data preprocessing completed. Shape:", processed.shape)

Data preprocessing completed. Shape: (1742, 80)


# 3. Feature Selection

## 3.1 Feature Selection

In [4]:
corr = pd.DataFrame(X_scaled, columns=X.columns).corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
to_drop = [col for col in upper.columns if any(upper[col] > 0.9)]
X_reduced = pd.DataFrame(X_scaled, columns=X.columns).drop(columns=to_drop)

print("Dropped highly correlated features:", to_drop)

Dropped highly correlated features: ['Total Bwd packets', 'Total Length of Fwd Packet', 'Total Length of Bwd Packet', 'Fwd Packet Length Max', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Packets/s', 'Flow IAT Std', 'Flow IAT Max', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Packet Length Max', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'Down/Up Ratio', 'Average Packet Size', 'Fwd Segment Size Avg', 'Bwd Segment Size Avg', 'Bwd Bytes/Bulk Avg', 'Bwd Packet/Bulk Avg', 'Bwd Bulk Rate Avg', 'Subflow Bwd Packets', 'FWD Init Win Bytes', 'Fwd Act Data Pkts', 'Active Max', 'Active Min', 'Idle Mean', 'Idle Max', 'Idle Min']


## 3.2 Random Forest importance

In [5]:
non_behavioral = ["Src Port", "Protocol"]
X_behavioral = X_reduced.drop(columns=non_behavioral, errors="ignore")

rf = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42
)
rf.fit(X_behavioral, y)

importances = (
    pd.Series(rf.feature_importances_, index=X_behavioral.columns)
    .sort_values(ascending=False)
)

# Select TOP 10 features
top_features = importances.head(10).index
X_selected = X_behavioral[top_features].copy()

# Add label back
X_selected["Label"] = y.values

# Save outputs
X_selected.to_csv("Data_selected.csv", index=False)
importances.to_csv("feature_importance.csv")

print("Top 10 selected behavioral features:")
for i, f in enumerate(top_features, 1):
    print(f"{i}. {f}")

Top 10 selected behavioral features:
1. Total Fwd Packet
2. Flow Bytes/s
3. Fwd IAT Std
4. Flow IAT Mean
5. Flow Duration
6. Bwd IAT Min
7. Bwd Init Win Bytes
8. Flow IAT Min
9. Subflow Fwd Packets
10. Subflow Bwd Bytes


# 4. Train/Test Split

In [6]:
X_final = X_selected.drop("Label", axis=1)
y_final = X_selected["Label"]

X_train, X_test, y_train, y_test = train_test_split(
    X_final, y_final, test_size=0.25, stratify=y_final, random_state=42
)

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)

Train shape: (1306, 10) Test shape: (436, 10)


# 5. Model Training

In [7]:
models = {
    "LogisticRegression": LogisticRegression(
        class_weight="balanced", max_iter=1000
    ),
    "RandomForest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=42
    )
}

trained_models = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    model.fit(X_train, y_train)
    trained_models[name] = model
    print(f"{name} training completed.")


Training LogisticRegression...
LogisticRegression training completed.

Training RandomForest...
RandomForest training completed.


# 6. Model Evaluation

In [8]:
for name, model in trained_models.items():
    print(f"\n=== {name} Evaluation ===")
    y_pred = model.predict(X_test)
    report = classification_report(y_test, y_pred, digits=4)
    bal_acc = balanced_accuracy_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    
    print(report)
    print("Balanced Accuracy:", bal_acc)
    print("Confusion Matrix:\n", cm)
    
    # Save model
    dump(model, f"results/model.joblib")

print("\nEvaluation completed.")


=== LogisticRegression Evaluation ===
              precision    recall  f1-score   support

           0     0.9975    1.0000    0.9988       407
           1     1.0000    0.9655    0.9825        29

    accuracy                         0.9977       436
   macro avg     0.9988    0.9828    0.9906       436
weighted avg     0.9977    0.9977    0.9977       436

Balanced Accuracy: 0.9827586206896552
Confusion Matrix:
 [[407   0]
 [  1  28]]

=== RandomForest Evaluation ===
              precision    recall  f1-score   support

           0     0.9975    1.0000    0.9988       407
           1     1.0000    0.9655    0.9825        29

    accuracy                         0.9977       436
   macro avg     0.9988    0.9828    0.9906       436
weighted avg     0.9977    0.9977    0.9977       436

Balanced Accuracy: 0.9827586206896552
Confusion Matrix:
 [[407   0]
 [  1  28]]

Evaluation completed.


# 7. Sample Testing

In [9]:
from joblib import load
import pandas as pd
import numpy as np

# Load saved model and scaler
rf_model = load("results/model.joblib")
scaler = load("results/scaler.joblib")

# Load new CSV
df_new = pd.read_csv("test/test.csv")

# Drop identifiers / non-feature columns
drop_cols = ["Flow ID", "Src IP", "Dst IP", "Timestamp", "Label"]
df_new.drop(columns=drop_cols, inplace=True, errors="ignore")

# Ensure all features that scaler expects exist
scaler_features = X.columns  # X from training
df_new_full = df_new.reindex(columns=scaler_features, fill_value=0)

# Make sure all columns are numeric
df_new_full = df_new_full.apply(pd.to_numeric, errors='coerce').fillna(0)

# Scale using the saved scaler
X_new_scaled_full = scaler.transform(df_new_full)

# Convert back to DataFrame with proper column names
X_new_scaled_full = pd.DataFrame(X_new_scaled_full, columns=scaler_features)

# Select only top features for prediction
X_new_selected = X_new_scaled_full[top_features]

# Predict
y_pred_new = rf_model.predict(X_new_selected)
print("Predictions on new CSV:", y_pred_new)

# Assuming y_pred_new is a list or NumPy array
row_numbers = [i + 1 for i, val in enumerate(y_pred_new) if val == 1]
print("Rows predicted as 1 (1-based):", row_numbers)


Predictions on new CSV: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 1 0 0 0 0 0 0 0]
Rows predicted as 1 (1-based): [19, 23]
